# Spherical covariance: build validity into the model

This lab constructs isotropic covariance functions on the unit sphere from non-negative Legendre coefficients. The validity claim comes from the spherical representation, not from numerical eigenvalue checks alone.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

root = Path.cwd()
while root != root.parent and not (root / 'implementations').exists():
    root = root.parent
sys.path.insert(0, str(root / 'implementations' / 'python' / 'spherical-covariance'))
from spherical_covariance import (
    latlon_to_unit_vectors, legendre_series_covariance,
    spherical_covariance_matrix, spherical_covariance_diagnostics,
)


## 1. Convert locations

The unit-vector representation avoids longitude wraparound in the distance calculation.

In [ ]:
lat = np.array([0, 0, 45, -35, 70, -60, 15, 82])
lon = np.array([0, 90, 45, 140, -120, 20, 179, -40])
sites = latlon_to_unit_vectors(lat, lon)
print('unit-vector norms:', np.linalg.norm(sites, axis=1))


## 2. Compare spectral coefficient choices

Each non-negative finite coefficient sequence defines a valid isotropic covariance on the two-sphere. Here we normalise each sequence to unit variance for comparison.

In [ ]:
theta = np.linspace(0, np.pi, 300)
coefficient_sets = {
    'low-degree emphasis': [0.65, 0.25, 0.10],
    'mixed degrees': [0.25, 0.25, 0.20, 0.15, 0.10, 0.05],
    'higher-degree emphasis': [0.05, 0.10, 0.15, 0.20, 0.25, 0.25],
}
for label, weights in coefficient_sets.items():
    plt.plot(theta, legendre_series_covariance(theta, weights, normalise=True), label=label)
plt.xlabel('Central angle (radians)')
plt.ylabel('Correlation')
plt.title('Valid isotropic covariance examples on the unit sphere')
plt.legend()
plt.grid(True)
plt.show()


## 3. Build and inspect a global covariance matrix

The eigenvalue check is a software diagnostic for this finite set of sites. The general validity guarantee comes from the non-negative Legendre coefficients.

In [ ]:
weights = [1.0, 0.35, 0.2, 0.1, 0.05]
C = spherical_covariance_matrix(sites, weights)
print(C.round(3))
print(spherical_covariance_diagnostics(C))
plt.imshow(C, interpolation='nearest')
plt.colorbar(label='Covariance')
plt.xlabel('Site index')
plt.ylabel('Site index')
plt.title('Spherical covariance matrix')
plt.show()


## 4. Inspect the coordinate edge cases

Duplicate coordinates and antipodal points are legitimate inputs. The model uses the central angle on a perfect sphere, not ellipsoidal geodesic distance.

In [ ]:
special = latlon_to_unit_vectors([0, 0, 0, 90, -90], [0, 0, 180, 0, 0])
C_special = spherical_covariance_matrix(special, [1.0, 0.4, 0.1])
print('duplicate-site covariance:', C_special[0, 1])
print('antipodal covariance:', C_special[0, 2])
print('diagnostics:', spherical_covariance_diagnostics(C_special))


## 5. Take the modelling boundary seriously

Do not use this scalar isotropic S2 model as an ellipsoidal geodesy engine, a network covariance model or a fitted Gaussian process. Each of those requires its own geometry, assumptions and inference method.